# 10 — ODIR Audit + True Zero-Shot External Validation

**NO TRAINING IN THIS NOTEBOOK.**

This notebook addresses the reviewer's ODIR concerns in the safest order:

1. Audit what ODIR files actually exist in the project.
2. Locate the ODIR annotation table and raw image files.
3. Recover **patient ID**, eye side, image filename, and eye-specific diagnostic keywords.
4. Construct a transparent **Cataract-v-Normal eye-level subset** from ODIR.
5. Confirm all ODIR images are external to the internal training dataset.
6. Evaluate the **final frozen MobileNetV2 baseline zero-shot** — no ODIR images are used for training.
7. Report external accuracy, sensitivity, specificity, AUC, confusion matrix, and patient-cluster bootstrap CIs.
8. Audit any old ODIR 70/30 adaptation artifacts for evidence of patient-level splitting.

## Important interpretation

- This notebook does **not** retrain or fine-tune on ODIR.
- Therefore the zero-shot result is genuine cross-dataset external testing.
- Multiple eyes from one patient may both appear in this external evaluation; that is not train/test leakage because **none of ODIR is used for training**.
- Confidence intervals are cluster-bootstrapped by patient ID when patient identifiers are available.
- The old 70/30 ODIR adaptation result is kept separate and must not be called external validation unless patient-level separation is verified.

In [ ]:
# ============================================================
# CELL 1 — SETUP
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import re
import json
import math
import random
import warnings

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)

PROJECT = Path('/content/drive/MyDrive/Cataract')

MODEL_DIR = (
    PROJECT
    / 'FINAL_REVISION_2026_08'
    / 'reviewer_10_2_clean_split'
    / 'MobileNetV2_frozen'
)

MODEL_PATH = MODEL_DIR / 'best.keras'

OUT = (
    PROJECT
    / 'FINAL_REVISION_2026_08'
    / 'odir_true_zeroshot'
)

OUT.mkdir(parents=True, exist_ok=True)

SEED = 42
BOOTSTRAPS = 5000
IMAGE_SIZE = (224, 224)

assert PROJECT.exists(), f'STOP: Missing project folder: {PROJECT}'
assert MODEL_PATH.exists(), f'STOP: Missing final MobileNetV2: {MODEL_PATH}'

print('TensorFlow:', tf.__version__)
print('Project:', PROJECT)
print('Final model:', MODEL_PATH)
print('Output:', OUT)

print('\n✅ CELL 1 COMPLETE')

In [ ]:
# ============================================================
# CELL 2 — AUDIT ODIR / EXTERNAL-VALIDATION FILES IN DRIVE
# ============================================================

KEYWORDS = (
    'odir',
    'external',
    'off-site',
    'offsite'
)

audit_rows = []

for p in PROJECT.rglob('*'):
    try:
        rel = str(p.relative_to(PROJECT))
    except Exception:
        rel = str(p)

    low = rel.lower()

    if any(k in low for k in KEYWORDS):
        audit_rows.append({
            'relative_path': rel,
            'name': p.name,
            'is_dir': p.is_dir(),
            'suffix': p.suffix.lower() if p.is_file() else '',
            'size_bytes': p.stat().st_size if p.is_file() else np.nan
        })

audit_df = pd.DataFrame(audit_rows)

audit_df.to_csv(
    OUT / 'ODIR_Drive_Audit_All_Matching_Paths.csv',
    index=False
)

print('ODIR/external-related paths found:', len(audit_df))

if len(audit_df):
    display(audit_df.head(300))
else:
    print('⚠️ No path containing ODIR/external keywords was found.')

print('\n✅ CELL 2 COMPLETE')

In [ ]:
# ============================================================
# CELL 3 — LOCATE / RECOVER THE REAL ODIR ANNOTATION TABLE
# ROBUST VERSION
# ============================================================

from pathlib import Path
import re
import os
import sys
import subprocess
import zipfile
import tarfile
import shutil
import numpy as np
import pandas as pd

TABULAR_SUFFIXES = {'.csv', '.xlsx', '.xls'}

# Important:
# Do NOT allow this notebook's own output CSV files to become
# "candidate ODIR source files".
OUT_RESOLVED = OUT.resolve()

RECOVERY_DIR = (
    PROJECT
    / 'FINAL_REVISION_2026_08'
    / 'ODIR_SOURCE_RECOVERED'
)

RECOVERY_DIR.mkdir(
    parents=True,
    exist_ok=True
)

EXTRACT_DIR = (
    RECOVERY_DIR
    / '_extracted_annotation_candidates'
)

EXTRACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def read_table(path):
    path = Path(path)

    if path.suffix.lower() == '.csv':
        # Try normal UTF-8 first, then common fallback.
        try:
            return pd.read_csv(path)
        except UnicodeDecodeError:
            return pd.read_csv(
                path,
                encoding='latin1'
            )

    return pd.read_excel(path)


def normalized_name(x):
    return re.sub(
        r'[^a-z0-9]+',
        '-',
        str(x).strip().lower()
    ).strip('-')


def normalized_columns(df):
    return {
        normalized_name(c): c
        for c in df.columns
    }


def odir_schema_score(df):
    """
    Score a table for the standard ODIR patient-level schema.

    Expected semantic fields:
    - patient ID
    - left fundus filename
    - right fundus filename
    - left diagnostic keywords
    - right diagnostic keywords
    """

    norm_names = list(
        normalized_columns(df).keys()
    )

    score = 0

    # Patient ID
    if any(
        c in norm_names
        for c in [
            'id',
            'patient-id',
            'patientid'
        ]
    ):
        score += 2

    # Fundus filenames
    if any(
        ('left' in c and 'fundus' in c)
        for c in norm_names
    ):
        score += 2

    if any(
        ('right' in c and 'fundus' in c)
        for c in norm_names
    ):
        score += 2

    # Diagnostic keywords
    if any(
        (
            'left' in c
            and 'diagnostic' in c
        )
        for c in norm_names
    ):
        score += 2

    if any(
        (
            'right' in c
            and 'diagnostic' in c
        )
        for c in norm_names
    ):
        score += 2

    return score


def is_inside_output(path):
    """
    Exclude files created by this notebook itself.
    """
    try:
        Path(path).resolve().relative_to(
            OUT_RESOLVED
        )
        return True
    except Exception:
        return False


def add_candidate(path, source, rows_out):
    """
    Read and score one candidate table.
    """

    path = Path(path)

    if not path.exists():
        return

    if path.suffix.lower() not in TABULAR_SUFFIXES:
        return

    if is_inside_output(path):
        return

    try:
        df = read_table(path)
        score = odir_schema_score(df)

        rows_out.append({
            'path': str(path),
            'source': source,
            'rows': len(df),
            'columns': len(df.columns),
            'schema_score': score,
            'column_names':
                ' | '.join(
                    map(str, df.columns)
                )
        })

    except Exception as e:
        rows_out.append({
            'path': str(path),
            'source': source,
            'rows': np.nan,
            'columns': np.nan,
            'schema_score': -1,
            'column_names':
                f'READ ERROR: '
                f'{type(e).__name__}: {e}'
        })


# ------------------------------------------------------------
# STEP 3A — Search ALL local Drive tables, not only paths whose
# names contain "ODIR" or "external".
# ------------------------------------------------------------

table_audit = []

local_tables = [
    p
    for p in PROJECT.rglob('*')
    if (
        p.is_file()
        and p.suffix.lower()
        in TABULAR_SUFFIXES
        and not is_inside_output(p)
    )
]

print(
    'Local CSV/XLS/XLSX files scanned:',
    len(local_tables)
)

for p in local_tables:
    add_candidate(
        p,
        source='local_drive',
        rows_out=table_audit
    )


# ------------------------------------------------------------
# STEP 3B — Search ZIP/TAR archives for a hidden annotation file.
# This is important because the original ODIR source may be
# stored inside an archive.
# ------------------------------------------------------------

archive_candidates = [
    p
    for p in PROJECT.rglob('*')
    if (
        p.is_file()
        and (
            p.suffix.lower() == '.zip'
            or p.name.lower().endswith(
                ('.tar', '.tar.gz', '.tgz')
            )
        )
        and not is_inside_output(p)
    )
]

print(
    'Archives scanned:',
    len(archive_candidates)
)

for archive_path in archive_candidates:

    try:

        # ZIP
        if archive_path.suffix.lower() == '.zip':

            with zipfile.ZipFile(
                archive_path,
                'r'
            ) as zf:

                for member in zf.namelist():

                    member_path = Path(member)

                    if (
                        member_path.suffix.lower()
                        not in TABULAR_SUFFIXES
                    ):
                        continue

                    low = member_path.name.lower()

                    # Prefer likely ODIR annotation names,
                    # but still allow schema validation later.
                    likely = (
                        'full_df' in low
                        or 'annotation' in low
                        or 'odir' in low
                    )

                    if not likely:
                        continue

                    safe_name = (
                        archive_path.stem
                        + '__'
                        + member_path.name
                    )

                    extracted = (
                        EXTRACT_DIR
                        / safe_name
                    )

                    with zf.open(member) as src_f, \
                         open(extracted, 'wb') as dst_f:
                        shutil.copyfileobj(
                            src_f,
                            dst_f
                        )

                    add_candidate(
                        extracted,
                        source=(
                            'archive:'
                            + str(archive_path)
                        ),
                        rows_out=table_audit
                    )

        # TAR / TAR.GZ / TGZ
        elif archive_path.name.lower().endswith(
            ('.tar', '.tar.gz', '.tgz')
        ):

            with tarfile.open(
                archive_path,
                'r:*'
            ) as tf:

                for member in tf.getmembers():

                    if not member.isfile():
                        continue

                    member_name = Path(
                        member.name
                    )

                    if (
                        member_name.suffix.lower()
                        not in TABULAR_SUFFIXES
                    ):
                        continue

                    low = member_name.name.lower()

                    likely = (
                        'full_df' in low
                        or 'annotation' in low
                        or 'odir' in low
                    )

                    if not likely:
                        continue

                    extracted_file = (
                        EXTRACT_DIR
                        / (
                            archive_path.stem
                            + '__'
                            + member_name.name
                        )
                    )

                    src_f = tf.extractfile(
                        member
                    )

                    if src_f is None:
                        continue

                    with src_f, open(
                        extracted_file,
                        'wb'
                    ) as dst_f:
                        shutil.copyfileobj(
                            src_f,
                            dst_f
                        )

                    add_candidate(
                        extracted_file,
                        source=(
                            'archive:'
                            + str(archive_path)
                        ),
                        rows_out=table_audit
                    )

    except Exception as e:
        print(
            'Archive scan warning:',
            archive_path.name,
            '->',
            type(e).__name__,
            str(e)[:200]
        )


# ------------------------------------------------------------
# STEP 3C — Evaluate what we have locally.
# ------------------------------------------------------------

table_audit_df = pd.DataFrame(
    table_audit
)

if len(table_audit_df):

    table_audit_df = (
        table_audit_df
        .drop_duplicates(
            subset=['path']
        )
        .sort_values(
            [
                'schema_score',
                'rows'
            ],
            ascending=[
                False,
                False
            ]
        )
        .reset_index(drop=True)
    )

else:

    table_audit_df = pd.DataFrame(
        columns=[
            'path',
            'source',
            'rows',
            'columns',
            'schema_score',
            'column_names'
        ]
    )


table_audit_df.to_csv(
    OUT
    / 'ODIR_Annotation_Candidate_Audit.csv',
    index=False
)

print(
    '\nTop local/archive candidates:'
)

display(
    table_audit_df.head(30)
)


strong = table_audit_df[
    table_audit_df[
        'schema_score'
    ] >= 8
].copy()


# ------------------------------------------------------------
# STEP 3D — If the original annotation is not in Drive,
# recover ONLY full_df.csv from the public Kaggle ODIR mirror.
#
# This is metadata only; it does NOT train anything and does NOT
# download the full image dataset here.
# ------------------------------------------------------------

if len(strong) == 0:

    print(
        '\nNo valid ODIR annotation table exists in the current Drive.'
    )

    print(
        'Attempting automatic recovery of full_df.csv '
        'from the public ODIR Kaggle mirror...'
    )

    try:

        subprocess.check_call(
            [
                sys.executable,
                '-m',
                'pip',
                'install',
                '-q',
                '-U',
                'kagglehub'
            ]
        )

        import kagglehub

        KAGGLE_HANDLE = (
            'andrewmvd/'
            'ocular-disease-recognition-odir5k'
        )

        kaggle_download_dir = (
            RECOVERY_DIR
            / 'kaggle_metadata'
        )

        kaggle_download_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        downloaded = (
            kagglehub.dataset_download(
                KAGGLE_HANDLE,
                path='full_df.csv',
                output_dir=str(
                    kaggle_download_dir
                )
            )
        )

        print(
            'KaggleHub returned:',
            downloaded
        )

        # Search the recovery directory because the API may
        # return either a file or a containing directory.
        recovered_tables = [
            p
            for p in kaggle_download_dir.rglob('*')
            if (
                p.is_file()
                and p.suffix.lower()
                in TABULAR_SUFFIXES
            )
        ]

        # Also consider the direct returned path.
        returned_path = Path(
            downloaded
        )

        if (
            returned_path.exists()
            and returned_path.is_file()
            and returned_path.suffix.lower()
            in TABULAR_SUFFIXES
        ):
            recovered_tables.append(
                returned_path
            )

        for p in recovered_tables:

            add_candidate(
                p,
                source='kaggle_public_mirror',
                rows_out=table_audit
            )

        # Rebuild audit after recovery.
        table_audit_df = pd.DataFrame(
            table_audit
        )

        table_audit_df = (
            table_audit_df
            .drop_duplicates(
                subset=['path']
            )
            .sort_values(
                [
                    'schema_score',
                    'rows'
                ],
                ascending=[
                    False,
                    False
                ]
            )
            .reset_index(drop=True)
        )

        table_audit_df.to_csv(
            OUT
            / 'ODIR_Annotation_Candidate_Audit.csv',
            index=False
        )

        strong = table_audit_df[
            table_audit_df[
                'schema_score'
            ] >= 8
        ].copy()

    except Exception as e:

        print(
            '\nAutomatic Kaggle annotation recovery failed:'
        )

        print(
            type(e).__name__,
            ':',
            e
        )


# ------------------------------------------------------------
# STEP 3E — Final source selection.
# ------------------------------------------------------------

if len(strong) == 0:

    (
        OUT
        / 'ODIR_SOURCE_BLOCKED.txt'
    ).write_text(
        'No valid ODIR annotation table was found locally, '
        'inside project archives, or through the automatic '
        'full_df.csv recovery attempt.\n'
        'No zero-shot result was generated.\n'
    )

    raise RuntimeError(
        '\nSTOP: The ODIR metadata source is still missing.\n'
        '\n'
        'This is NOT a model/training error.\n'
        '\n'
        'Required source:\n'
        '  full_df.csv\n'
        'OR\n'
        '  ODIR-5K_Training_Annotations(Updated)_V2.xlsx\n'
        '\n'
        'The table must contain patient ID, left/right fundus '
        'filenames, and left/right diagnostic keywords.'
    )


# Prefer the strongest schema match.
# For ties, prefer:
#   1) local source
#   2) archive source
#   3) downloaded mirror
# and a plausible patient-level row count.

source_priority = {
    'local_drive': 0,
    'kaggle_public_mirror': 2
}


def priority_for_source(s):

    s = str(s)

    if s == 'local_drive':
        return 0

    if s.startswith('archive:'):
        return 1

    if s == 'kaggle_public_mirror':
        return 2

    return 9


strong['source_priority'] = (
    strong['source']
    .apply(
        priority_for_source
    )
)

strong['plausible_patient_rows'] = (
    pd.to_numeric(
        strong['rows'],
        errors='coerce'
    )
    .between(
        3000,
        6000
    )
)

strong = strong.sort_values(
    [
        'schema_score',
        'plausible_patient_rows',
        'source_priority'
    ],
    ascending=[
        False,
        False,
        True
    ]
)


ANNOTATION_PATH = Path(
    strong.iloc[0]['path']
)

annotation = read_table(
    ANNOTATION_PATH
)


# ------------------------------------------------------------
# STEP 3F — Hard verification before moving to Cell 4.
# ------------------------------------------------------------

final_score = odir_schema_score(
    annotation
)

if final_score < 8:

    raise RuntimeError(
        'STOP: Selected table failed the final ODIR schema verification.'
    )


print('\n========================================')
print('✅ VALID ODIR ANNOTATION TABLE FOUND')
print('========================================')

print(
    'Source:',
    strong.iloc[0]['source']
)

print(
    'Path:',
    ANNOTATION_PATH
)

print(
    'Rows:',
    len(annotation)
)

print(
    'Columns:',
    len(annotation.columns)
)

print(
    'Schema score:',
    final_score,
    '/ 10'
)

print('\nColumn names:')

for c in annotation.columns:
    print(' -', c)


# Save final source manifest for reproducibility.
pd.DataFrame(
    [{
        'annotation_path':
            str(ANNOTATION_PATH),
        'source':
            strong.iloc[0]['source'],
        'rows':
            len(annotation),
        'columns':
            len(annotation.columns),
        'schema_score':
            final_score
    }]
).to_csv(
    OUT
    / 'ODIR_Selected_Annotation_Source.csv',
    index=False
)


print('\n✅ CELL 3 COMPLETE')
print('NEXT: Run CELL 4.')

In [ ]:
# ============================================================
# CELL 4 — MAP STANDARD ODIR COLUMNS + BUILD EYE-LEVEL LABEL TABLE
# ============================================================

def find_col(columns, include_terms, exact_candidates=()):

    for exact in exact_candidates:
        for c in columns:
            if str(c).strip().lower() == exact.lower():
                return c

    for c in columns:
        low = str(c).strip().lower()
        if all(term.lower() in low for term in include_terms):
            return c

    return None


cols = list(annotation.columns)

ID_COL = find_col(
    cols,
    [],
    exact_candidates=('ID', 'Patient ID', 'PatientID')
)

LEFT_FILE_COL = find_col(
    cols,
    ('left', 'fundus')
)

RIGHT_FILE_COL = find_col(
    cols,
    ('right', 'fundus')
)

LEFT_DIAG_COL = find_col(
    cols,
    ('left', 'diagnostic')
)

RIGHT_DIAG_COL = find_col(
    cols,
    ('right', 'diagnostic')
)

mapped = {
    'ID_COL': ID_COL,
    'LEFT_FILE_COL': LEFT_FILE_COL,
    'RIGHT_FILE_COL': RIGHT_FILE_COL,
    'LEFT_DIAG_COL': LEFT_DIAG_COL,
    'RIGHT_DIAG_COL': RIGHT_DIAG_COL
}

print('Mapped columns:')
for k, v in mapped.items():
    print(k, '->', v)

if any(v is None for v in mapped.values()):
    raise RuntimeError(
        'STOP: Could not unambiguously map all standard ODIR columns.'
    )


def classify_eye_keyword(keyword):

    s = str(keyword).strip().lower()

    # Eye-specific cataract label.
    if 'cataract' in s:
        return 'Cataract'

    # Standard ODIR normal keyword.
    if s in {
        'normal fundus',
        'normal',
        'normal fundus photo'
    }:
        return 'Normal'

    # Conservative normal handling:
    # require normal wording and reject obvious disease words.
    disease_terms = [
        'cataract',
        'glaucoma',
        'diabetic',
        'retin',
        'macular',
        'hypertens',
        'myopia',
        'degeneration',
        'occlusion',
        'hemorr',
        'haemorr',
        'drusen',
        'atrophy',
        'scar',
        'edema',
        'oedema'
    ]

    if 'normal' in s and not any(term in s for term in disease_terms):
        return 'Normal'

    return 'Exclude'


eye_rows = []

for _, row in annotation.iterrows():

    pid = str(row[ID_COL]).strip()

    for side, file_col, diag_col in [
        ('Left', LEFT_FILE_COL, LEFT_DIAG_COL),
        ('Right', RIGHT_FILE_COL, RIGHT_DIAG_COL)
    ]:

        filename = str(row[file_col]).strip()
        diagnostic = str(row[diag_col]).strip()

        if (
            filename == ''
            or filename.lower() in {'nan', 'none'}
        ):
            continue

        label = classify_eye_keyword(
            diagnostic
        )

        eye_rows.append({
            'patient_id': pid,
            'eye_side': side,
            'filename': Path(filename).name,
            'diagnostic_keyword': diagnostic,
            'external_label': label
        })

eye_df = pd.DataFrame(
    eye_rows
)

eye_df.to_csv(
    OUT / 'ODIR_All_Eye_Level_Annotation.csv',
    index=False
)

eligible = eye_df[
    eye_df['external_label'].isin(
        ['Cataract', 'Normal']
    )
].copy()

print('All annotated eyes:', len(eye_df))
print('\nEye-level label counts:')
print(eye_df['external_label'].value_counts(dropna=False))

print('\nEligible zero-shot clinical subset:')
print(eligible['external_label'].value_counts())
print('Total eligible:', len(eligible))
print('Distinct patients:', eligible['patient_id'].nunique())

eligible.to_csv(
    OUT / 'ODIR_Eligible_Cataract_Normal_Before_File_Match.csv',
    index=False
)

print('\n✅ CELL 4 COMPLETE')

In [ ]:
# ============================================================
# CELL 5 — FINAL ROBUST ODIR IMAGE RECOVERY + UNIQUE-IMAGE MATCHING
# ============================================================

from pathlib import Path
import os
import sys
import subprocess
import shutil
import numpy as np
import pandas as pd

IMAGE_SUFFIXES = {
    '.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'
}

KAGGLE_HANDLE = (
    'andrewmvd/'
    'ocular-disease-recognition-odir5k'
)


# ------------------------------------------------------------
# STEP 5A — NORMALISE / AUDIT THE ELIGIBLE ANNOTATION TABLE
# ------------------------------------------------------------

eligible_work = eligible.copy()

eligible_work['filename'] = (
    eligible_work['filename']
    .astype(str)
    .str.strip()
    .map(lambda x: Path(x).name)
)

eligible_work['filename_key'] = (
    eligible_work['filename']
    .str.lower()
)

eligible_work['patient_id'] = (
    eligible_work['patient_id']
    .astype(str)
    .str.strip()
)

eligible_work['external_label'] = (
    eligible_work['external_label']
    .astype(str)
    .str.strip()
)

print('Raw eligible annotation rows:', len(eligible_work))
print(
    'Unique eligible filenames:',
    eligible_work['filename_key'].nunique()
)


# ------------------------------------------------------------
# Check whether one image filename has conflicting labels.
# If it does, the image cannot safely be used.
# ------------------------------------------------------------

label_counts = (
    eligible_work
    .groupby('filename_key')['external_label']
    .nunique()
)

conflicting_label_keys = set(
    label_counts[
        label_counts > 1
    ].index
)

label_conflicts = (
    eligible_work[
        eligible_work['filename_key']
        .isin(conflicting_label_keys)
    ]
    .sort_values(
        ['filename_key', 'external_label']
    )
)

label_conflicts.to_csv(
    OUT / 'ODIR_Annotation_Label_Conflicts.csv',
    index=False
)

print(
    'Filenames with conflicting Cataract/Normal labels:',
    len(conflicting_label_keys)
)

if conflicting_label_keys:

    print(
        '\n❌ Conflicting labels were found for the same ODIR image filename.'
    )

    display(
        label_conflicts.head(30)
    )

    raise RuntimeError(
        'STOP: The same ODIR image filename has conflicting '
        'Cataract/Normal labels. Resolve the annotation source '
        'before external evaluation.'
    )


# ------------------------------------------------------------
# Check whether one filename maps to multiple patient IDs.
# ------------------------------------------------------------

patient_counts = (
    eligible_work
    .groupby('filename_key')['patient_id']
    .nunique()
)

multi_patient_keys = set(
    patient_counts[
        patient_counts > 1
    ].index
)

multi_patient_rows = (
    eligible_work[
        eligible_work['filename_key']
        .isin(multi_patient_keys)
    ]
    .sort_values(
        ['filename_key', 'patient_id']
    )
)

multi_patient_rows.to_csv(
    OUT / 'ODIR_Annotation_MultiPatient_Filename_Audit.csv',
    index=False
)

print(
    'Filenames associated with >1 patient ID:',
    len(multi_patient_keys)
)

if multi_patient_keys:

    print(
        '\n❌ One or more ODIR image filenames map to multiple patient IDs.'
    )

    display(
        multi_patient_rows.head(30)
    )

    raise RuntimeError(
        'STOP: ODIR filename-to-patient mapping is ambiguous.'
    )


# ------------------------------------------------------------
# Collapse repeated annotation rows to ONE ROW PER IMAGE.
# This prevents duplicate counting in the external test.
# ------------------------------------------------------------

duplicate_annotation_rows = (
    eligible_work[
        eligible_work.duplicated(
            subset=['filename_key'],
            keep=False
        )
    ]
    .sort_values(
        ['filename_key', 'patient_id']
    )
)

duplicate_annotation_rows.to_csv(
    OUT / 'ODIR_Duplicate_Annotation_Rows.csv',
    index=False
)

eligible_unique = (
    eligible_work
    .sort_values(
        [
            'filename_key',
            'patient_id',
            'eye_side'
        ]
    )
    .drop_duplicates(
        subset=['filename_key'],
        keep='first'
    )
    .reset_index(drop=True)
)

eligible_unique.to_csv(
    OUT / 'ODIR_Eligible_Unique_Images.csv',
    index=False
)

print(
    'Duplicate annotation rows collapsed:',
    len(eligible_work) - len(eligible_unique)
)

print(
    'Final unique images to evaluate:',
    len(eligible_unique)
)

print(
    '\nUnique-image class counts:'
)

display(
    eligible_unique[
        'external_label'
    ]
    .value_counts()
    .rename_axis('Class')
    .reset_index(name='Images')
)


# ------------------------------------------------------------
# STEP 5B — SEARCH EXISTING PROJECT IMAGES
# ------------------------------------------------------------

def collect_images(root):

    root = Path(root)

    if not root.exists():
        return []

    if root.is_file():

        if root.suffix.lower() in IMAGE_SUFFIXES:
            return [root]

        return []

    return [
        p
        for p in root.rglob('*')
        if (
            p.is_file()
            and p.suffix.lower()
            in IMAGE_SUFFIXES
        )
    ]


eligible_keys = set(
    eligible_unique[
        'filename_key'
    ]
)

project_images = collect_images(
    PROJECT
)

project_match_count = sum(
    1
    for p in project_images
    if p.name.lower() in eligible_keys
)

print(
    '\nProject image files scanned:',
    len(project_images)
)

print(
    'Matching ODIR filenames already in project:',
    project_match_count
)


# ------------------------------------------------------------
# STEP 5C — GET THE REAL KAGGLE DATASET ROOT
#
# IMPORTANT FIX:
# kagglehub may use the Colab cache and return a path such as
# /kaggle/input/ocular-disease-recognition-odir5k.
# We MUST scan the path returned by dataset_download().
# ------------------------------------------------------------

subprocess.check_call(
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        '-U',
        'kagglehub'
    ]
)

import kagglehub

print(
    '\nResolving public ODIR dataset through KaggleHub...'
)

downloaded_path = (
    kagglehub.dataset_download(
        KAGGLE_HANDLE,
        force_download=False
    )
)

KAGGLE_DATASET_ROOT = Path(
    downloaded_path
)

print(
    'KaggleHub returned dataset root:',
    KAGGLE_DATASET_ROOT
)

print(
    'Dataset root exists:',
    KAGGLE_DATASET_ROOT.exists()
)

if not KAGGLE_DATASET_ROOT.exists():

    raise RuntimeError(
        'STOP: KaggleHub returned a dataset path that does not exist: '
        f'{KAGGLE_DATASET_ROOT}'
    )


# Show top-level structure for reproducibility/debugging.
if KAGGLE_DATASET_ROOT.is_dir():

    top_level = sorted(
        list(
            KAGGLE_DATASET_ROOT.iterdir()
        ),
        key=lambda p: p.name.lower()
    )

    print(
        '\nTop-level ODIR dataset contents:'
    )

    for p in top_level[:50]:
        print(
            ' -',
            p.name,
            '[DIR]' if p.is_dir() else '[FILE]'
        )


# ------------------------------------------------------------
# STEP 5D — SCAN THE ACTUAL RETURNED KAGGLE ROOT
# ------------------------------------------------------------

kaggle_images = collect_images(
    KAGGLE_DATASET_ROOT
)

print(
    '\nImages found under returned Kaggle root:',
    len(kaggle_images)
)

kaggle_match_count = sum(
    1
    for p in kaggle_images
    if p.name.lower() in eligible_keys
)

print(
    'Eligible ODIR filename matches in Kaggle root:',
    kaggle_match_count
)


# Helpful diagnostics only if there are unexpectedly zero matches.
if kaggle_match_count == 0:

    actual_examples = [
        p.name
        for p in kaggle_images[:20]
    ]

    expected_examples = (
        eligible_unique[
            'filename'
        ]
        .head(20)
        .tolist()
    )

    print(
        '\nExample expected annotation filenames:'
    )
    print(
        expected_examples
    )

    print(
        '\nExample actual Kaggle image filenames:'
    )
    print(
        actual_examples
    )

    raise RuntimeError(
        'STOP: The returned Kaggle dataset root contains images, '
        'but none of their basenames match the ODIR annotation filenames. '
        'This indicates an annotation-source mismatch rather than a model issue.'
    )


# ------------------------------------------------------------
# STEP 5E — BUILD IMAGE INDEX FROM THE REAL KAGGLE ROOT
# + project fallback
# ------------------------------------------------------------

candidate_image_paths = list(
    dict.fromkeys(
        kaggle_images
        + project_images
    )
)

basename_index = {}

for p in candidate_image_paths:

    key = p.name.lower()

    if key not in eligible_keys:
        continue

    basename_index.setdefault(
        key,
        []
    ).append(
        p
    )


# ------------------------------------------------------------
# Prefer a consistent packaged representation.
# ------------------------------------------------------------

def image_source_rank(path):

    low = str(path).lower()

    if 'preprocessed_images' in low:
        return 0

    if (
        'training images' in low
        or 'training_images' in low
        or '/training/' in low
    ):
        return 1

    # Any file under the returned Kaggle dataset root.
    try:
        Path(path).resolve().relative_to(
            KAGGLE_DATASET_ROOT.resolve()
        )
        return 2
    except Exception:
        pass

    return 3


def describe_source_rank(rank):

    return {
        0:
            'Kaggle ODIR preprocessed_images',
        1:
            'Kaggle ODIR raw/training images',
        2:
            'other image under Kaggle ODIR dataset root',
        3:
            'project-local fallback image'
    }.get(
        int(rank),
        'unknown'
    )


# ------------------------------------------------------------
# STEP 5F — MATCH ONE FILE TO EACH UNIQUE EXTERNAL IMAGE
# ------------------------------------------------------------

resolved_rows = []
missing_files = []
multiple_source_rows = []

for _, r in eligible_unique.iterrows():

    key = r[
        'filename_key'
    ]

    matches = basename_index.get(
        key,
        []
    )

    if len(matches) == 0:

        missing_files.append(
            r['filename']
        )

        continue

    matches_sorted = sorted(
        matches,
        key=lambda p: (
            image_source_rank(p),
            len(str(p)),
            str(p).lower()
        )
    )

    chosen = matches_sorted[0]

    rr = r.to_dict()

    rr['filepath'] = str(
        chosen
    )

    rr['actual_image_filename'] = (
        chosen.name
    )

    rr['image_source_rank'] = int(
        image_source_rank(
            chosen
        )
    )

    rr[
        'number_of_same_basename_candidates'
    ] = int(
        len(matches)
    )

    resolved_rows.append(
        rr
    )

    if len(matches) > 1:

        multiple_source_rows.append({
            'filename':
                r['filename'],
            'chosen':
                str(chosen),
            'all_candidates':
                ' | '.join(
                    map(
                        str,
                        matches_sorted
                    )
                )
        })


resolved = pd.DataFrame(
    resolved_rows
)

missing_unique = sorted(
    set(
        missing_files
    )
)

multiple_sources_df = pd.DataFrame(
    multiple_source_rows
)


# ------------------------------------------------------------
# STEP 5G — SAVE MATCHING AUDIT
# ------------------------------------------------------------

pd.DataFrame(
    {
        'missing_filename':
            missing_unique
    }
).to_csv(
    OUT
    / 'ODIR_Missing_Annotated_Files.csv',
    index=False
)

multiple_sources_df.to_csv(
    OUT
    / 'ODIR_Ambiguous_File_Matches.csv',
    index=False
)

resolved.to_csv(
    OUT
    / 'ODIR_Resolved_ZeroShot_File_Index.csv',
    index=False
)


# ------------------------------------------------------------
# STEP 5H — STRICT COVERAGE CHECK ON UNIQUE IMAGES
# ------------------------------------------------------------

coverage = (
    len(resolved)
    / len(eligible_unique)
    if len(eligible_unique)
    else 0.0
)

print(
    '\n========================================'
)

print(
    'FINAL ODIR UNIQUE-IMAGE MATCHING AUDIT'
)

print(
    '========================================'
)

print(
    'Raw eligible annotation rows:',
    len(eligible_work)
)

print(
    'Unique eligible images:',
    len(eligible_unique)
)

print(
    'Resolved unique images:',
    len(resolved)
)

print(
    'Missing unique images:',
    len(missing_unique)
)

print(
    'Images with >1 available source copy:',
    len(multiple_source_rows)
)

print(
    f'Unique-image coverage: '
    f'{100 * coverage:.2f}%'
)


if len(resolved):

    source_summary = (
        resolved[
            'image_source_rank'
        ]
        .value_counts()
        .sort_index()
        .rename_axis(
            'image_source_rank'
        )
        .reset_index(
            name='images'
        )
    )

    source_summary[
        'source_description'
    ] = source_summary[
        'image_source_rank'
    ].map(
        describe_source_rank
    )

    print(
        '\nChosen image source summary:'
    )

    display(
        source_summary
    )


if coverage < 0.98:

    print(
        '\n❌ Matching is below the required 98% threshold.'
    )

    print(
        'First missing filenames:'
    )

    print(
        missing_unique[:30]
    )

    raise RuntimeError(
        'STOP: ODIR unique-image matching coverage is '
        f'{100*coverage:.2f}%, below the required 98%.'
    )


# ------------------------------------------------------------
# STEP 5I — FINAL SANITY CHECKS
# ------------------------------------------------------------

assert resolved[
    'filename_key'
].nunique() == len(resolved), (
    'STOP: Duplicate image filenames remain after matching.'
)

assert not resolved[
    'filepath'
].duplicated().any(), (
    'STOP: The same physical image filepath is mapped to '
    'more than one external test record.'
)

assert set(
    resolved[
        'external_label'
    ].unique()
).issubset({
    'Cataract',
    'Normal'
})


dominant_rank = int(
    resolved[
        'image_source_rank'
    ]
    .mode()
    .iloc[0]
)

dominant_source = (
    describe_source_rank(
        dominant_rank
    )
)


pd.DataFrame(
    [{
        'dataset_handle':
            KAGGLE_HANDLE,
        'kagglehub_returned_root':
            str(
                KAGGLE_DATASET_ROOT
            ),
        'raw_eligible_annotation_rows':
            len(eligible_work),
        'unique_eligible_images':
            len(eligible_unique),
        'resolved_unique_images':
            len(resolved),
        'coverage':
            coverage,
        'dominant_source_rank':
            dominant_rank,
        'dominant_source':
            dominant_source
    }]
).to_csv(
    OUT
    / 'ODIR_Image_Source_Manifest.csv',
    index=False
)


print(
    '\n========================================'
)

print(
    '✅ ODIR UNIQUE-IMAGE MATCHING PASSED'
)

print(
    '========================================'
)

print(
    'Kaggle dataset root:',
    KAGGLE_DATASET_ROOT
)

print(
    'Unique external images:',
    len(resolved)
)

print(
    'Dominant source:',
    dominant_source
)

print(
    f'Coverage: {100*coverage:.2f}%'
)

print(
    '\n✅ CELL 5 COMPLETE'
)

print(
    'NEXT: Run CELL 6.'
)

In [ ]:
# ============================================================
# CELL 6 — CHECK INTERNAL-v-ODIR FILE SEPARATION
# ============================================================

INTERNAL = PROJECT / 'Data_Clean_LeakageControlled_FINAL'

internal_names = set()

if INTERNAL.exists():
    internal_names = {
        p.name
        for p in INTERNAL.rglob('*')
        if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES
    }

odir_names = set(
    resolved['filename'].astype(str)
)

same_basename = sorted(
    internal_names.intersection(
        odir_names
    )
)

pd.DataFrame(
    {'same_basename_internal_odir': same_basename}
).to_csv(
    OUT / 'ODIR_Internal_Basename_Overlap_Audit.csv',
    index=False
)

print('Internal image basenames:', len(internal_names))
print('ODIR zero-shot basenames:', len(odir_names))
print('Same basename count:', len(same_basename))

if same_basename:
    print(
        '⚠️ Same basenames were found. This does NOT prove identical images, '
        'but they must be checked before claiming source independence.'
    )
else:
    print('✅ No internal/ODIR basename overlap found.')

print('\n✅ CELL 6 COMPLETE')

In [ ]:
# ============================================================
# CELL 7 — LOAD FINAL MOBILENETV2 + ZERO-SHOT PREDICTION
# ============================================================

model = tf.keras.models.load_model(
    MODEL_PATH,
    compile=False
)

print('Loaded final MobileNetV2.')
print('Model layers:', len(model.layers))

pred_probs = []
bad_images = []

for i, row in resolved.reset_index(drop=True).iterrows():

    fp = row['filepath']

    try:
        img = tf.keras.utils.load_img(
            fp,
            target_size=IMAGE_SIZE,
            color_mode='rgb',
            interpolation='nearest'
        )

        arr = tf.keras.utils.img_to_array(
            img
        ).astype(np.float32) / 255.0

        p = model(
            np.expand_dims(arr, axis=0),
            training=False
        ).numpy()[0]

        pred_probs.append(p)

    except Exception as e:
        pred_probs.append(
            [np.nan, np.nan, np.nan]
        )
        bad_images.append({
            'filepath': fp,
            'error': f'{type(e).__name__}: {e}'
        })

pred_probs = np.asarray(
    pred_probs,
    dtype=float
)

pd.DataFrame(
    bad_images
).to_csv(
    OUT / 'ODIR_Bad_Image_Reads.csv',
    index=False
)

good_mask = np.isfinite(
    pred_probs
).all(axis=1)

print('Predicted images:', int(good_mask.sum()))
print('Bad/unreadable images:', int((~good_mask).sum()))

if good_mask.mean() < 0.995:
    raise RuntimeError(
        'STOP: Too many ODIR images failed prediction.'
    )

eval_df = resolved.reset_index(
    drop=True
).loc[good_mask].copy()

probs_good = pred_probs[
    good_mask
]

eval_df['P_Cataract'] = probs_good[:, 0]
eval_df['P_Normal'] = probs_good[:, 1]
eval_df['P_NotEye'] = probs_good[:, 2]

denom = (
    eval_df['P_Cataract'].to_numpy()
    + eval_df['P_Normal'].to_numpy()
)

clinical_score = np.divide(
    eval_df['P_Cataract'].to_numpy(),
    denom,
    out=np.full(len(eval_df), 0.5, dtype=float),
    where=denom > 0
)

eval_df['Clinical_Cataract_Score'] = clinical_score

eval_df['y_true_binary'] = (
    eval_df['external_label']
    == 'Cataract'
).astype(int)

eval_df['y_pred_binary'] = (
    eval_df['Clinical_Cataract_Score']
    >= 0.5
).astype(int)

eval_df['three_class_pred'] = np.argmax(
    probs_good,
    axis=1
)

eval_df['three_class_pred_name'] = (
    eval_df['three_class_pred']
    .map({
        0: 'Cataract',
        1: 'Normal',
        2: 'Not Eye'
    })
)

eval_df.to_csv(
    OUT / 'ODIR_ZeroShot_Predictions.csv',
    index=False
)

print('\n✅ ZERO-SHOT PREDICTIONS COMPLETE')
print('✅ CELL 7 COMPLETE')

In [ ]:
# ============================================================
# CELL 8 — EXTERNAL ZERO-SHOT METRICS
# ============================================================

y = eval_df[
    'y_true_binary'
].to_numpy(dtype=int)

pred = eval_df[
    'y_pred_binary'
].to_numpy(dtype=int)

score = eval_df[
    'Clinical_Cataract_Score'
].to_numpy(dtype=float)

cm = confusion_matrix(
    y,
    pred,
    labels=[0, 1]
)

TN, FP, FN, TP = cm.ravel()

accuracy = (
    TP + TN
) / len(y)

sensitivity = (
    TP / (TP + FN)
    if (TP + FN) else np.nan
)

specificity = (
    TN / (TN + FP)
    if (TN + FP) else np.nan
)

auc = roc_auc_score(
    y,
    score
)

noteye_rate = float(
    (
        eval_df['three_class_pred_name']
        == 'Not Eye'
    ).mean()
)

summary = {
    'External_N': int(len(y)),
    'Distinct_Patients': int(
        eval_df['patient_id'].nunique()
    ),
    'Cataract_N': int(y.sum()),
    'Normal_N': int((y == 0).sum()),
    'Accuracy': float(accuracy),
    'Sensitivity': float(sensitivity),
    'Specificity': float(specificity),
    'AUC': float(auc),
    'TN': int(TN),
    'FP': int(FP),
    'FN': int(FN),
    'TP': int(TP),
    'ThreeClass_NotEye_Prediction_Rate': noteye_rate
}

print('========================================')
print('TRUE ZERO-SHOT ODIR RESULT')
print('========================================')

for k, v in summary.items():
    print(f'{k}: {v}')

print('\nConfusion matrix [Normal, Cataract]:')
print(cm)

print('\n✅ CELL 8 COMPLETE')

In [ ]:
# ============================================================
# CELL 9 — PATIENT-CLUSTER BOOTSTRAP 95% CIs
# ============================================================

rng = np.random.default_rng(
    SEED
)

patients = eval_df[
    'patient_id'
].astype(str).unique()

patient_to_indices = {
    pid: np.where(
        eval_df['patient_id'].astype(str).to_numpy()
        == pid
    )[0]
    for pid in patients
}

boot_acc = []
boot_sens = []
boot_spec = []
boot_auc = []

for _ in range(BOOTSTRAPS):

    sampled_patients = rng.choice(
        patients,
        size=len(patients),
        replace=True
    )

    idx_parts = [
        patient_to_indices[pid]
        for pid in sampled_patients
    ]

    idx = np.concatenate(
        idx_parts
    )

    yy = y[idx]
    pp = pred[idx]
    ss = score[idx]

    # Need both classes for AUC.
    if len(np.unique(yy)) < 2:
        continue

    b_cm = confusion_matrix(
        yy,
        pp,
        labels=[0, 1]
    )

    b_TN, b_FP, b_FN, b_TP = b_cm.ravel()

    boot_acc.append(
        (b_TP + b_TN) / len(yy)
    )

    boot_sens.append(
        b_TP / (b_TP + b_FN)
        if (b_TP + b_FN)
        else np.nan
    )

    boot_spec.append(
        b_TN / (b_TN + b_FP)
        if (b_TN + b_FP)
        else np.nan
    )

    boot_auc.append(
        roc_auc_score(
            yy,
            ss
        )
    )


def ci(values):
    arr = np.asarray(
        values,
        dtype=float
    )

    arr = arr[
        np.isfinite(arr)
    ]

    return (
        float(np.percentile(arr, 2.5)),
        float(np.percentile(arr, 97.5))
    )


acc_ci = ci(boot_acc)
sens_ci = ci(boot_sens)
spec_ci = ci(boot_spec)
auc_ci = ci(boot_auc)

summary.update({
    'Accuracy_CI_Low': acc_ci[0],
    'Accuracy_CI_High': acc_ci[1],
    'Sensitivity_CI_Low': sens_ci[0],
    'Sensitivity_CI_High': sens_ci[1],
    'Specificity_CI_Low': spec_ci[0],
    'Specificity_CI_High': spec_ci[1],
    'AUC_CI_Low': auc_ci[0],
    'AUC_CI_High': auc_ci[1],
    'Bootstrap_Replicates': BOOTSTRAPS,
    'Bootstrap_Unit': 'patient_id'
})

summary_df = pd.DataFrame(
    [summary]
)

summary_df.to_csv(
    OUT / 'ODIR_True_ZeroShot_Summary.csv',
    index=False
)

with open(
    OUT / 'ODIR_True_ZeroShot_Summary.json',
    'w'
) as f:
    json.dump(
        summary,
        f,
        indent=2
    )

print('95% patient-cluster bootstrap CIs:')
print('Accuracy:', acc_ci)
print('Sensitivity:', sens_ci)
print('Specificity:', spec_ci)
print('AUC:', auc_ci)

print('\n✅ CELL 9 COMPLETE')

In [ ]:
# ============================================================
# CELL 10 — AUDIT OLD 70/30 ODIR ADAPTATION ARTIFACTS
# ============================================================

external_dirs = []

for p in PROJECT.rglob('*'):
    if p.is_dir():
        low = str(p).lower()
        if ('odir' in low) or ('external_validation' in low):
            external_dirs.append(p)

artifact_rows = []

for d in external_dirs:
    for p in d.rglob('*'):
        if p.is_file():
            artifact_rows.append({
                'path': str(p),
                'name': p.name,
                'suffix': p.suffix.lower(),
                'size_bytes': p.stat().st_size
            })

old_artifacts = pd.DataFrame(
    artifact_rows
).drop_duplicates(
    subset=['path']
)

old_artifacts.to_csv(
    OUT / 'ODIR_Old_Adaptation_Artifact_Audit.csv',
    index=False
)

print('Old ODIR/external artifact files found:', len(old_artifacts))

if len(old_artifacts):
    display(old_artifacts.head(300))

# Look for obvious train/test CSVs and patient-ID columns.
split_candidate_rows = []

for path_str in old_artifacts[
    old_artifacts['suffix'].isin(
        ['.csv', '.xlsx', '.xls']
    )
]['path'].tolist():

    p = Path(path_str)

    try:
        df = read_table(p)

        lower_cols = [
            str(c).strip().lower()
            for c in df.columns
        ]

        patient_cols = [
            c for c in df.columns
            if (
                'patient' in str(c).lower()
                or str(c).strip().lower() == 'id'
            )
        ]

        split_candidate_rows.append({
            'path': str(p),
            'rows': len(df),
            'columns': ' | '.join(map(str, df.columns)),
            'patient_id_columns': ' | '.join(map(str, patient_cols)),
            'looks_train': 'train' in p.name.lower(),
            'looks_test': 'test' in p.name.lower(),
            'looks_split': 'split' in p.name.lower()
        })

    except Exception:
        pass

split_audit = pd.DataFrame(
    split_candidate_rows
)

split_audit.to_csv(
    OUT / 'ODIR_Old_Split_Table_Audit.csv',
    index=False
)

if len(split_audit):
    print('\nPotential old split tables:')
    display(split_audit)

# Do not claim patient-level separation automatically.
patient_split_verified = False

if len(split_audit):
    train_tables = split_audit[
        split_audit['looks_train']
    ]

    test_tables = split_audit[
        split_audit['looks_test']
    ]

    # Verification only if exactly one obvious train and one obvious test table
    # both expose a patient-ID column.
    if len(train_tables) == 1 and len(test_tables) == 1:

        tr_path = Path(train_tables.iloc[0]['path'])
        te_path = Path(test_tables.iloc[0]['path'])

        tr = read_table(tr_path)
        te = read_table(te_path)

        tr_patient_cols = [
            c for c in tr.columns
            if (
                'patient' in str(c).lower()
                or str(c).strip().lower() == 'id'
            )
        ]

        te_patient_cols = [
            c for c in te.columns
            if (
                'patient' in str(c).lower()
                or str(c).strip().lower() == 'id'
            )
        ]

        if len(tr_patient_cols) == 1 and len(te_patient_cols) == 1:

            train_ids = set(
                tr[tr_patient_cols[0]]
                .astype(str)
                .str.strip()
            )

            test_ids = set(
                te[te_patient_cols[0]]
                .astype(str)
                .str.strip()
            )

            overlap = sorted(
                train_ids.intersection(
                    test_ids
                )
            )

            pd.DataFrame(
                {'patient_id_overlap': overlap}
            ).to_csv(
                OUT / 'ODIR_Old_Train_Test_Patient_Overlap.csv',
                index=False
            )

            patient_split_verified = (
                len(overlap) == 0
            )

print('\nOld 70/30 patient-level split automatically verified:',
      patient_split_verified)

if not patient_split_verified:
    print(
        '⚠️ Do NOT claim the old 70/30 ODIR adaptation was patient-level '
        'unless separate source evidence is found.'
    )

print('\n✅ CELL 10 COMPLETE')

In [ ]:
# ============================================================
# CELL 11 — MANUSCRIPT-READY DRAFT + FINAL DECISION
# ============================================================

paper_text = f'''
Zero-shot external validation was performed on an eye-level
Cataract-v-Normal subset of ODIR using the final internally trained
MobileNetV2 model without any ODIR fine-tuning. The external analysis
included {summary["External_N"]} images from
{summary["Distinct_Patients"]} distinct patients
({summary["Cataract_N"]} cataract and {summary["Normal_N"]} normal eyes).
Using the renormalized Cataract-versus-Normal probability at a 0.5
threshold, accuracy was {100*summary["Accuracy"]:.2f}%,
sensitivity {100*summary["Sensitivity"]:.2f}%,
specificity {100*summary["Specificity"]:.2f}%, and AUC
{summary["AUC"]:.4f}. Patient-cluster bootstrap confidence intervals
were calculated using {BOOTSTRAPS} resamples to account for correlation
between left and right eyes from the same individual. No ODIR image was
used for training or model selection, so this result is reported as
zero-shot cross-dataset external evaluation.
'''.strip()

(
    OUT / 'ODIR_ZeroShot_Manuscript_Draft.txt'
).write_text(
    paper_text
)

limitations_text = (
    'The historical ODIR 70/30 adaptation experiment must remain '
    'separate from the zero-shot external evaluation. '
    + (
        'Available split artifacts showed no patient-ID overlap.'
        if patient_split_verified
        else
        'The available saved artifacts did not unambiguously verify '
        'patient-level train/test separation, so that adapted result '
        'should not be described as confirmed patient-independent validation.'
    )
)

(
    OUT / 'ODIR_Adaptation_Limitation_Draft.txt'
).write_text(
    limitations_text
)

print('========================================')
print('ZERO-SHOT MANUSCRIPT DRAFT')
print('========================================')
print(paper_text)

print('\n========================================')
print('OLD ADAPTATION LIMITATION')
print('========================================')
print(limitations_text)

print('\n✅ CELL 11 COMPLETE')

In [ ]:
# ============================================================
# CELL 12 — FINAL COMPLETION CHECK
# ============================================================

required = [
    'ODIR_Drive_Audit_All_Matching_Paths.csv',
    'ODIR_Annotation_Candidate_Audit.csv',
    'ODIR_All_Eye_Level_Annotation.csv',
    'ODIR_Eligible_Cataract_Normal_Before_File_Match.csv',
    'ODIR_Resolved_ZeroShot_File_Index.csv',
    'ODIR_ZeroShot_Predictions.csv',
    'ODIR_True_ZeroShot_Summary.csv',
    'ODIR_True_ZeroShot_Summary.json',
    'ODIR_Old_Adaptation_Artifact_Audit.csv',
    'ODIR_Old_Split_Table_Audit.csv',
    'ODIR_ZeroShot_Manuscript_Draft.txt',
    'ODIR_Adaptation_Limitation_Draft.txt'
]

missing = [
    fn for fn in required
    if not (OUT / fn).exists()
]

if missing:
    print('Missing outputs:')
    for fn in missing:
        print('❌', fn)

    raise RuntimeError(
        'STOP: ODIR audit/zero-shot output is incomplete.'
    )

(
    OUT / 'ODIR_ZEROSHOT_DONE.txt'
).write_text(
    'ODIR source audit and true zero-shot external validation completed.\n'
    'No ODIR training or fine-tuning performed.\n'
    f'External N={summary["External_N"]}\n'
    f'Distinct patients={summary["Distinct_Patients"]}\n'
    f'Accuracy={summary["Accuracy"]:.8f}\n'
    f'Sensitivity={summary["Sensitivity"]:.8f}\n'
    f'Specificity={summary["Specificity"]:.8f}\n'
    f'AUC={summary["AUC"]:.8f}\n'
    f'Old adaptation patient split verified={patient_split_verified}\n'
)

print('========================================')
print('✅ ODIR TRUE ZERO-SHOT EVALUATION COMPLETE')
print('========================================')

print('\nSaved to:')
print(OUT)

print(
    '\nNEXT: Download/save this executed notebook and upload it to ChatGPT.'
)

print(
    '\nDO NOT TRAIN ON ODIR. '
    'Do not modify the manuscript until these results are interpreted.'
)